In [1]:
from pathlib import Path

import anndata as ad
import ovrlpy
import pandas as pd

/dh-projects/ag-ishaque/analysis/muellni/envs/ovrlpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_folder = Path("/dh-projects/ag-ishaque/raw_data/tiesmeys-ovrlpy/Xenium-brain-2024")

In [4]:
def gene_counts(df):
    return (
        df.groupby(["gene", "cell_id"], observed=True)
        .size()
        .to_frame("n")
        .reset_index("gene")
        .pivot(columns="gene", values="n")
        .fillna(0)
        .astype("int16")
    )

In [5]:
for i in [2, 3]:
    replicate = data_folder / f"replicate{i}"

    transcripts = ovrlpy.io.read_Xenium(
        replicate / "transcripts.parquet",
        additional_columns=["overlaps_nucleus", "cell_id"],
    )
    transcripts = ovrlpy.process_coordinates(transcripts)

    nuclear_df = transcripts.to_pandas().loc[lambda df: df["overlaps_nucleus"] == 1]

    ad.AnnData(
        gene_counts(nuclear_df[nuclear_df["z"] < nuclear_df["z_center"]])
    ).write_h5ad(f"bottom_count_mtx_replicate{i}.h5ad", compression="gzip")
    ad.AnnData(
        gene_counts(nuclear_df[nuclear_df["z"] > nuclear_df["z_center"]])
    ).write_h5ad(f"top_count_mtx_replicate{i}.h5ad", compression="gzip")

/dh-projects/ag-ishaque/analysis/muellni/envs/ovrlpy/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/dh-projects/ag-ishaque/analysis/muellni/envs/ovrlpy/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Run mapmycells and proceed with results

In [22]:
from zipfile import ZipFile


def read_mapmycells_zip(path: Path):
    with ZipFile(path) as zipfile:
        with zipfile.open(path.stem + ".csv") as csvfile:
            df = pd.read_csv(csvfile, index_col=0, comment="#")
    return df


def global_coherence(file1, file2):
    mapmycells_df1 = read_mapmycells_zip(Path(file1))
    mapmycells_df2 = read_mapmycells_zip(Path(file2))

    mapmycells_df2["family_name"] = (
        mapmycells_df2["class_name"].str.split(" ").str[-1].astype("category")
    )
    mapmycells_df1["family_name"] = (
        mapmycells_df1["class_name"].str.split(" ").str[-1].astype("category")
    )

    cells_in_both_slices = mapmycells_df2.loc[
        lambda df: df.index.isin(mapmycells_df1.index)
    ].index
    mapmycells_df2 = mapmycells_df2.loc[cells_in_both_slices]
    mapmycells_df1 = mapmycells_df1.loc[cells_in_both_slices]

    # ratio of coherently predicted cell types across the vertical demarcation
    ct_label_top = mapmycells_df2["class_name"]
    ct_label_bottom = mapmycells_df1["class_name"]

    return (ct_label_top == ct_label_bottom).mean()

In [23]:
coherence_rep2 = global_coherence(
    "bottom_count_mtx_replicate2_10xWholeMouseBrain(CCN20230722)_CorrelationMapping_UTC_1752765646794.zip",
    "top_count_mtx_replicate2_10xWholeMouseBrain(CCN20230722)_CorrelationMapping_UTC_1752765421920.zip",
)
coherence_rep3 = global_coherence(
    "bottom_count_mtx_replicate3_10xWholeMouseBrain(CCN20230722)_CorrelationMapping_UTC_1752765159601.zip",
    "top_count_mtx_replicate3_10xWholeMouseBrain(CCN20230722)_CorrelationMapping_UTC_1752765452196.zip",
)

print(f"Replicate2: coherence {coherence_rep2:.2f}")
print(f"Replicate3: coherence {coherence_rep3:.2f}")

Replicate2: coherence 0.80
Replicate3: coherence 0.79
